In [5]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ Client ready")

✅ Client ready


In [2]:
!python -m pip install openai pydub python-dotenv

Defaulting to user installation because normal site-packages is not writeable


# Step 3: Basic Transcription (Without Chunking)

This section loads a short audio file, sends it to Whisper, and prints the raw response plus the extracted text.

In [7]:
# Path to the audio file to transcribe
# The sample file is in the notebook root and is named Arthur.mp3
audio_file = "Arthur.mp3"

print("File exists:", os.path.exists(audio_file))

with open(audio_file, "rb") as f:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=f
    )

print("\nFull response object:")
print(transcription)

print("\nTranscribed text:")
print(transcription.text)


File exists: True

Full response object:
Transcription(text="The story of Arthur the Rat. Once upon a time, there was a young rat who couldn't make up his mind. Whenever the other rats asked him if he would like to come out hunting with them, he would answer in a hoarse voice, I don't know. And when they said, would you rather stay inside, he would say yes or no either. He'd always shake making a choice. One fine day, his Aunt Josephine said to him, now look here, no one will ever care for you if you carry on like this. You have no more mind of your own than a greasy old blade of grass. The young rat coughed and looked quiet as usual, but said nothing. Don't you think so, said his aunt, stamping her foot, for she couldn't bear to see the young rat so cold-blooded. I don't know, was all he ever answered. And then he walked off to think for an hour or more, whether he would stay in his hole in the ground or go out into the loft. One night the rats heard a loud noise in the loft. It was a

# Step 4: Transcription with Prompts (Guided Approach)

This step adds a prompt to the transcription request so Whisper has context about the audio topic, names, and expected style. The prompt can improve accuracy for technical terms and speaker names.

In [8]:
# Guided prompt for the transcription
prompt_text = (
    "This audio is a short English conversation about a technical topic. "
    "The speakers use clear phrasing and may mention software, data, or machine learning terms. "
    "Transcribe carefully and preserve punctuation."
)

audio_file = "Arthur.mp3"
output_file = "output/transcription_prompted.txt"

print("Using prompt:", prompt_text)
print("File exists:", os.path.exists(audio_file))

with open(audio_file, "rb") as f:
    transcription_with_prompt = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        prompt=prompt_text
    )

print("\nPrompted response object:")
print(transcription_with_prompt)

print("\nPrompted transcribed text:")
print(transcription_with_prompt.text)

# Save prompted transcription for later review
os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, "w", encoding="utf-8") as out:
    out.write(transcription_with_prompt.text)

print("Saved prompted transcription to:", output_file)

# Optional comparison if Step 3 transcription exists
if 'transcription' in globals():
    print("\n--- Comparison ---")
    print("Step 3 text length:", len(transcription.text))
    print("Step 4 text length:", len(transcription_with_prompt.text))

Using prompt: This audio is a short English conversation about a technical topic. The speakers use clear phrasing and may mention software, data, or machine learning terms. Transcribe carefully and preserve punctuation.
File exists: True

Prompted response object:
Transcription(text="The story of Arthur the Rat. Once upon a time there was a young rat who couldn't make up his mind. Whenever the other rats asked him if he would like to come out hunting with them, he would answer in a hoarse voice, I don't know. And when they said, would you rather stay inside, he would say yes or no either. He'd always shake making a choice. One fine day his Aunt Josephine said to him, now look here, no one will ever care for you if you carry on like this. You have no more mind of your own than a greasy old blade of grass. The young rat coughed and looked quiet as usual but said nothing. Don't you think so, said his aunt, stamping her foot, for she couldn't bear to see the young rat so cold-blooded. I do

# Step 5: Transcription Without Prompts (Unguided Approach)

This step transcribes the same audio file again without providing any additional prompt. The goal is to compare the raw transcription against the prompted one and analyze differences.

In [9]:
# Transcription without any prompt
audio_file = "Arthur.mp3"
output_file = "output/transcription_unguided.txt"

print("File exists:", os.path.exists(audio_file))

with open(audio_file, "rb") as f:
    transcription_unguided = client.audio.transcriptions.create(
        model="whisper-1",
        file=f
    )

print("\nUnguided response object:")
print(transcription_unguided)

print("\nUnguided transcribed text:")
print(transcription_unguided.text)

os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, "w", encoding="utf-8") as out:
    out.write(transcription_unguided.text)

print("Saved unguided transcription to:", output_file)

if 'transcription_with_prompt' in globals():
    print("\n--- Prompted vs Unguided Comparison ---")
    print("Prompted text length:", len(transcription_with_prompt.text))
    print("Unguided text length:", len(transcription_unguided.text))
    print("\nPrompted text preview:\n", transcription_with_prompt.text[:400])
    print("\nUnguided text preview:\n", transcription_unguided.text[:400])
else:
    print("Prompted transcription is not available yet for comparison.")

File exists: True

Unguided response object:
Transcription(text="The story of Arthur the Rat. Once upon a time, there was a young rat who couldn't make up his mind. Whenever the other rats asked him if he would like to come out hunting with them, he would answer in a hoarse voice, I don't know. And when they said, would you rather stay inside, he would say yes or no either. He'd always shake making a choice. One fine day, his Aunt Josephine said to him, now look here, no one will ever care for you if you carry on like this. You have no more mind of your own than a greasy old blade of grass. The young rat coughed and looked quiet as usual, but said nothing. Don't you think so, said his aunt, stamping her foot, for she couldn't bear to see the young rat so cold-blooded. I don't know, was all he ever answered. And then he walked off to think for an hour or more, whether he would stay in his hole in the ground or go out into the loft. One night the rats heard a loud noise in the loft. It w

# Step 6: Implementing Audio Chunking

This step splits the long audio into smaller 10-minute chunks, processes each chunk separately, and writes a combined transcript with timestamps.

In [3]:
import math
import os
import shutil
import subprocess

# Use ffmpeg/ffprobe for chunking, since pydub depends on audioop which is missing in this Python build.
ffmpeg_path = shutil.which("ffmpeg") or shutil.which("ffmpeg.exe")
ffprobe_path = shutil.which("ffprobe") or shutil.which("ffprobe.exe")
print("ffmpeg path:", ffmpeg_path)
print("ffprobe path:", ffprobe_path)
if ffmpeg_path is None or ffprobe_path is None:
    raise RuntimeError("ffmpeg and ffprobe are required for chunking. Install FFmpeg and add it to PATH.")

if "client" not in globals():
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        print("Initialized OpenAI client from OPENAI_API_KEY.")
    else:
        raise RuntimeError("OpenAI client is not initialized. Run the earlier setup cell first.")

audio_file = "Arthur.mp3"
chunk_length_s = 10 * 60  # 10 minutes in seconds
output_dir = "chunks"
combined_output_file = "output/transcription_chunks_combined.txt"

if not os.path.exists(audio_file):
    raise FileNotFoundError(f"Audio file not found: {audio_file}")
print("File exists:", os.path.exists(audio_file))

# Get audio duration using ffprobe
duration_cmd = [
    ffprobe_path,
    "-v",
    "error",
    "-show_entries",
    "format=duration",
    "-of",
    "default=noprint_wrappers=1:nokey=1",
    audio_file,
]
try:
    duration_output = subprocess.check_output(duration_cmd, stderr=subprocess.STDOUT, text=True).strip()
    duration_s = float(duration_output)
except subprocess.CalledProcessError as err:
    raise RuntimeError(f"ffprobe failed: {err.output}") from err
except ValueError:
    raise RuntimeError(f"Unable to parse duration from ffprobe output: {duration_output!r}")

print("Audio duration (seconds):", duration_s)
chunk_count = math.ceil(duration_s / chunk_length_s)
if chunk_count == 0:
    raise RuntimeError("No audio duration detected; cannot create chunks.")

os.makedirs(output_dir, exist_ok=True)
chunks = []
for i in range(chunk_count):
    start_s = i * chunk_length_s
    duration_chunk = min(chunk_length_s, duration_s - start_s)
    chunk_filename = os.path.join(output_dir, f"Arthur_chunk_{i+1:02d}.mp3")
    chunk_cmd = [
        ffmpeg_path,
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-ss",
        str(start_s),
        "-i",
        audio_file,
        "-t",
        str(duration_chunk),
        "-c",
        "copy",
        chunk_filename,
    ]
    subprocess.run(chunk_cmd, check=True)
    chunks.append((chunk_filename, start_s, start_s + duration_chunk))
    print(f"Created chunk {i+1}: {chunk_filename} ({start_s:.1f}s - {start_s + duration_chunk:.1f}s)")

print("Total chunks created:", len(chunks))

# Transcribe each chunk and combine results
os.makedirs(os.path.dirname(combined_output_file), exist_ok=True)
all_results = []
for chunk_filename, start_s, end_s in chunks:
    with open(chunk_filename, "rb") as f:
        chunk_transcription = client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
        )
    text = chunk_transcription.text.strip()
    all_results.append({
        "chunk_file": chunk_filename,
        "start_s": start_s,
        "end_s": end_s,
        "text": text,
    })
    print(f"Transcribed {chunk_filename}: {len(text)} characters")

with open(combined_output_file, "w", encoding="utf-8") as out:
    for result in all_results:
        out.write(f"[{result['start_s']:.1f}s - {result['end_s']:.1f}s]\n")
        out.write(result["text"] + "\n\n")

print("Saved combined chunked transcription to:", combined_output_file)

ffmpeg path: C:\Users\madhu\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1-full_build\bin\ffmpeg.EXE
ffprobe path: C:\Users\madhu\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1-full_build\bin\ffprobe.EXE
Initialized OpenAI client from OPENAI_API_KEY.
File exists: True
Audio duration (seconds): 228.362429
Created chunk 1: chunks\Arthur_chunk_01.mp3 (0.0s - 228.4s)
Total chunks created: 1
Transcribed chunks\Arthur_chunk_01.mp3: 2940 characters
Saved combined chunked transcription to: output/transcription_chunks_combined.txt


# Step 7: Transcribing Chunks with Timestamps

This step transcribes each previously created chunk, adjusts the timestamps by the chunk offset, and writes a combined transcript with chunk-level timestamps.


In [ ]:
timestamped_output_file = "output/transcription_chunks_timestamped.txt"

if 'chunks' not in globals():
    raise RuntimeError("Chunk metadata is missing. Run the chunk creation cell first.")

if 'client' not in globals():
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
    else:
        raise RuntimeError("OpenAI client is not initialized. Run the earlier setup cell first.")

print("Transcribing chunks with adjusted timestamps...")
timestamped_results = []
for chunk_filename, start_s, end_s in chunks:
    if not os.path.exists(chunk_filename):
        raise FileNotFoundError(f"Chunk file not found: {chunk_filename}")
    with open(chunk_filename, "rb") as f:
        chunk_transcription = client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
        )
    text = chunk_transcription.text.strip()
    timestamped_results.append({
        "chunk_file": chunk_filename,
        "start_s": start_s,
        "end_s": end_s,
        "text": text,
    })
    print(f"Chunk {chunk_filename} transcribed, offset start={start_s:.1f}s")

with open(timestamped_output_file, "w", encoding="utf-8") as out:
    for result in timestamped_results:
        out.write(f"[{result['start_s']:.1f}s - {result['end_s']:.1f}s]\n")
        out.write(result["text"] + "\n\n")

print("Saved combined timestamped transcription to:", timestamped_output_file)


# Step 7: Transcribing Chunks with Timestamps

This step transcribes each previously created chunk, adjusts the timestamps by the chunk offset, and writes a combined transcript with chunk-level timestamps.


In [ ]:
timestamped_output_file = "output/transcription_chunks_timestamped.txt"

if 'chunks' not in globals():
    raise RuntimeError("Chunk metadata is missing. Run the chunk creation cell first.")

if 'client' not in globals():
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
    else:
        raise RuntimeError("OpenAI client is not initialized. Run the earlier setup cell first.")

print("Transcribing chunks with adjusted timestamps...")
timestamped_results = []
for chunk_filename, start_s, end_s in chunks:
    if not os.path.exists(chunk_filename):
        raise FileNotFoundError(f"Chunk file not found: {chunk_filename}")
    with open(chunk_filename, "rb") as f:
        chunk_transcription = client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
        )
    text = chunk_transcription.text.strip()
    timestamped_results.append({
        "chunk_file": chunk_filename,
        "start_s": start_s,
        "end_s": end_s,
        "text": text,
    })
    print(f"Chunk {chunk_filename} transcribed, offset start={start_s:.1f}s")

with open(timestamped_output_file, "w", encoding="utf-8") as out:
    for result in timestamped_results:
        out.write(f"[{result['start_s']:.1f}s - {result['end_s']:.1f}s]\n")
        out.write(result["text"] + "\n\n")

print("Saved combined timestamped transcription to:", timestamped_output_file)


# Step 8: Exporting with Timestamps

This step exports the chunked transcription into multiple readable formats: text, SRT subtitle, and JSON.


In [ ]:
from pathlib import Path
import json
import os

readable_output_file = 'output/transcription_chunks_readable.txt'
srt_output_file = 'output/transcription_chunks.srt'
json_output_file = 'output/transcription_chunks.json'
timestamped_output_file = 'output/transcription_chunks_timestamped.txt'

def format_srt_timestamp(seconds):
    ms = int(round((seconds - int(seconds)) * 1000))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def load_timestamped_results():
    if 'timestamped_results' in globals():
        return timestamped_results
    if Path(timestamped_output_file).exists():
        items = []
        lines = Path(timestamped_output_file).read_text(encoding='utf-8').splitlines()
        i = 0
        while i < len(lines):
            if lines[i].startswith('[') and ']' in lines[i]:
                times = lines[i].strip('[]').split(' - ')
                start_s = float(times[0].rstrip('s'))
                end_s = float(times[1].rstrip('s'))
                i += 1
                text_lines = []
                while i < len(lines) and lines[i].strip() != '':
                    text_lines.append(lines[i])
                    i += 1
                items.append({
                    'chunk_file': None,
                    'start_s': start_s,
                    'end_s': end_s,
                    'text': '\n'.join(text_lines).strip(),
                })
            else:
                i += 1
        return items
    if 'all_results' in globals():
        return all_results
    raise RuntimeError('Unable to find transcription results. Run Step 7 or create the timestamped file first.')

results = load_timestamped_results()
if not results:
    raise RuntimeError('No transcription segments found to export.')

os.makedirs('output', exist_ok=True)

with open(readable_output_file, 'w', encoding='utf-8') as out:
    for item in results:
        out.write(f"[{item['start_s']:.1f}s - {item['end_s']:.1f}s]
")
        out.write(item['text'] + '\n\n')

with open(srt_output_file, 'w', encoding='utf-8') as out:
    for index, item in enumerate(results, start=1):
        start_ts = format_srt_timestamp(item['start_s'])
        end_ts = format_srt_timestamp(item['end_s'])
        out.write(f"{index}\n{start_ts} --> {end_ts}\n{item['text']}\n\n")

with open(json_output_file, 'w', encoding='utf-8') as out:
    json.dump(results, out, indent=2, ensure_ascii=False)

print('Exported readable text to:', readable_output_file)
print('Exported subtitle file to:', srt_output_file)
print('Exported JSON file to:', json_output_file)


# Step 8: Exporting with Timestamps

This step exports the chunked transcription into multiple readable formats: text, SRT subtitle, and JSON.


In [ ]:
from pathlib import Path
import json
import os

readable_output_file = 'output/transcription_chunks_readable.txt'
srt_output_file = 'output/transcription_chunks.srt'
json_output_file = 'output/transcription_chunks.json'
timestamped_output_file = 'output/transcription_chunks_timestamped.txt'

def format_srt_timestamp(seconds):
    ms = int(round((seconds - int(seconds)) * 1000))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

def load_timestamped_results():
    if 'timestamped_results' in globals():
        return timestamped_results
    if Path(timestamped_output_file).exists():
        items = []
        lines = Path(timestamped_output_file).read_text(encoding='utf-8').splitlines()
        i = 0
        while i < len(lines):
            if lines[i].startswith('[') and ']' in lines[i]:
                times = lines[i].strip('[]').split(' - ')
                start_s = float(times[0].rstrip('s'))
                end_s = float(times[1].rstrip('s'))
                i += 1
                text_lines = []
                while i < len(lines) and lines[i].strip() != '':
                    text_lines.append(lines[i])
                    i += 1
                items.append({
                    'chunk_file': None,
                    'start_s': start_s,
                    'end_s': end_s,
                    'text': '\n'.join(text_lines).strip(),
                })
            else:
                i += 1
        return items
    if 'all_results' in globals():
        return all_results
    raise RuntimeError('Unable to find transcription results. Run Step 7 or create the timestamped file first.')

results = load_timestamped_results()
if not results:
    raise RuntimeError('No transcription segments found to export.')

os.makedirs('output', exist_ok=True)

with open(readable_output_file, 'w', encoding='utf-8') as out:
    for item in results:
        out.write(f"[{item['start_s']:.1f}s - {item['end_s']:.1f}s]
")
        out.write(item['text'] + '\n\n')

with open(srt_output_file, 'w', encoding='utf-8') as out:
    for index, item in enumerate(results, start=1):
        start_ts = format_srt_timestamp(item['start_s'])
        end_ts = format_srt_timestamp(item['end_s'])
        out.write(f"{index}\n{start_ts} --> {end_ts}\n{item['text']}\n\n")

with open(json_output_file, 'w', encoding='utf-8') as out:
    json.dump(results, out, indent=2, ensure_ascii=False)

print('Exported readable text to:', readable_output_file)
print('Exported subtitle file to:', srt_output_file)
print('Exported JSON file to:', json_output_file)
